# 05 — Feature-importance tables and figures
Extract all 132 descriptor importances from trusted final model bundles selected through their metadata. **Inputs:** notebook 03's models and metadata. **Outputs:** complete importance CSVs plus Figure 6 (RF) and Figures S3–S4 (LightGBM).

RF uses mean decrease in impurity. LightGBM uses its default split-count importance; the generic extraction filename retains `tree_mdi` for compatibility and does not imply identical measures. Top 15 descriptors and their order come from the inorganic domain; COD O/M values use the same descriptor list. MP figures show the inorganic model only.

In labels, μ denotes the atomic-fraction-weighted mean and d the mean absolute deviation. No external baseline CSV is required by the selected publication figures.


In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from project_config import make_cfg
from private_paths import COD_DIR, MP_DIR
cfg_cod = make_cfg(project_name="COD_CANON_V2", root_cif=COD_DIR)
cfg_mp = make_cfg(project_name="MP_ALL", root_cif=MP_DIR)
PAPER_RUN = "PAPER_V2"
RUN_OVERRIDES = {}  # (project_name, model, domain): exact run_id if selection is ambiguous.
runs = {model: f"{PAPER_RUN}__final__{model}__magpie" for model in ("rf", "lgbm")}

import pandas as pd
from IPython.display import display
from reports.paper_inputs import prepare_report_inputs
from reports.importance import importance_table
from reports.plot_importance import plot_feature_importance_for_cfg
selection = prepare_report_inputs([cfg_cod, cfg_mp], runs, overrides=RUN_OVERRIDES,
                                  paper_run=PAPER_RUN, kind="model")
display(pd.DataFrame(selection))


## Complete feature tables
Retain every descriptor before selecting the displayed top 15.


In [ ]:
for cfg in (cfg_cod, cfg_mp):
    display(importance_table(cfg, runs=runs, domains=["N", "O", "M"],
                             top_n=None, save=True, verbose=True))


## Publication panels
Use a shared inorganic ranking for each project/model combination; omit the pooled “Other” bar and panel titles.


In [ ]:
for cfg in (cfg_cod, cfg_mp):
    for model in ("rf", "lgbm"):
        table = plot_feature_importance_for_cfg(cfg, run_id=runs[model], model=model,
            domains=["I", "M", "O"] if cfg is cfg_cod else ["I"], top_k=15,
            save=True, title="", plot_other=False, use_hatch=False,
            name=f"feature_importance__{model}__run_{runs[model]}_notitle")
        display(table)
